# 31. Multiclass, multilabel, and multioutput tasks
Multiclass classification chooses one of several labels. Multilabel classification can assign several labels to the same example. Multioutput regression predicts several numbers. Their target shapes and evaluation meanings differ. A wrapper can fit a compatible base estimator separately for each target, but this may ignore relationships among outputs.

Inspect target meaning and shape before selecting a wrapper. A multilabel indicator table uses one column per label. Subset accuracy requires the entire predicted label set to match, which is stricter than per-label accuracy. Compare micro and macro metrics and support. For multioutput regression, decide how individual output errors should be aggregated and retain units.

In [ ]:
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
X, y = make_multilabel_classification(n_samples=120, n_features=10, n_classes=3, random_state=42) # Generate label-indicator targets.
Xt, Xv, yt, yv = train_test_split(X, y, random_state=42) # Create a development validation split.
model = OneVsRestClassifier(LogisticRegression(max_iter=1000)).fit(Xt, yt) # Fit one binary predictor per label.
pred = model.predict(Xv) # Return one indicator per sample and label.
print("Target shape:", y.shape) # Three possible labels per example.
print("Micro F1:", round(f1_score(yv, pred, average="micro"), 3)) # Aggregate label decisions.
print("Subset accuracy:", round(accuracy_score(yv, pred), 3)) # Require every label to match.
assert pred.shape == yv.shape # Preserve output dimensions.

## One-versus-rest ROC and precision–recall
For each class, compare that class against all remaining classes. The classifier’s probability columns must match classes_. Turn y into an indicator table with label_binarize using that same order. ROC and precision–recall curves take scores, not already thresholded labels. Macro averages give equal weight to classes; prevalence affects precision–recall behavior. The number of curve thresholds can differ because thresholds are based on distinct scores. Do not average ROC AUCs by passing an arbitrary label order or report a class with no positive validation examples.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
X, y = load_wine(return_X_y=True) # A three-class task.
Xt, Xv, yt, yv = train_test_split(X, y, stratify=y, random_state=42) # Development-only evaluation split.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xt, yt) # Fit without validation rows.
p = model.predict_proba(Xv) # Columns follow model.classes_.
Y = label_binarize(yv, classes=model.classes_) # Align one-versus-rest targets.
for j, label in enumerate(model.classes_):
    fpr, tpr, thresholds = roc_curve(Y[:, j], p[:, j]) # Class j versus all others.
    precision, recall, pr_thresholds = precision_recall_curve(Y[:, j], p[:, j]) # Use the same continuous scores.
    print("Class", label, "AUC", round(roc_auc_score(Y[:, j], p[:, j]), 3), "AP", round(average_precision_score(Y[:, j], p[:, j]), 3)) # Report per-class ranking.
print("Macro OVR AUC:", round(roc_auc_score(yv, p, multi_class="ovr", average="macro", labels=model.classes_), 3)) # Explicit strategy and class order.
assert Y.shape == p.shape # Every probability column has a matching target.

### Reading the result
Each class receives its own AUC and average precision. AUC summarizes ROC ranking; AP summarizes precision across recall changes. The example compares classes, not thresholds for deployment. Do not tune a threshold for each class and claim mutually exclusive labels without defining how conflicting decisions are resolved.

### Transfer exercise
Add a multilabel classification_report using the original Chapter 31 indicator targets. Compare micro, macro, weighted and samples F1.

## Your turn
Construct one example target for each of multiclass, multilabel, and multioutput regression. Explain why blindly flattening a two-dimensional target loses meaning.

Try the task before opening the solution on the course website.